In [1]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt

def plot_velocity_hist(pcd, bins: int = 50, range=(-5, 5)):
    vel = np.asarray(pcd[:, 3], dtype=np.float64).ravel()
    vel = vel[np.isfinite(vel)]

    mean = float(vel.mean())
    std = float(vel.std())

    fig, ax = plt.subplots()
    counts, bin_edges, patches = ax.hist(vel, bins=bins, range=range)

    ax.set_xlabel("v_r")
    ax.set_ylabel("Count")

    ax.axvline(mean, linestyle="--", color="red", linewidth=1)
    ymax = ax.get_ylim()[1]
    ax.text(mean, 0.96 * ymax,
            f"mean={mean:.3f}\nstd={std:.3f}",
            ha="center", va="top", color="red")

    ax.set_xticks(bin_edges)
    ax.set_xticklabels([f"{edge:.1f}" for edge in bin_edges], rotation=90, fontsize=8)

    fig.tight_layout()
    plt.show()




def visualize_xy_points(points, title):

    if points.shape[1] >= 2:
        xy = points[:, :2]

    x, y = xy[:, 0], xy[:, 1]

    plt.figure(figsize=(15, 10))
    plt.scatter(x, y, s=0.5, c="blue", alpha=0.6)  
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(title)
    # plt.axis("equal") 
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()


def visualize_dynamic_points(points, title):
    """
    可视化点云的动态与静止点 (忽略 z 轴，只显示 x,y)
    
    Args:
        points (np.ndarray): 点云数组 (N,4)，最后一维是速度
        title (str): 图标题
    """
    
    # 动态点 (v > 1)
    dynamic_points = points[np.abs(points[:, 3]) > 1]
    # 静止点 (v <= 1)
    static_points = points[np.abs(points[:, 3]) <= 1]
    
    plt.figure(figsize=(15, 5))
    
    if static_points.size > 0:
        plt.scatter(static_points[:, 0], static_points[:, 1], 
                    s=0.5, c="gray", alpha=0.5, label="Static (v≤0.5)")
    if dynamic_points.size > 0:
        plt.scatter(dynamic_points[:, 0], dynamic_points[:, 1], 
                    s=0.5, c="red", alpha=0.7, label="Dynamic (v>0.5)")
    
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(title)
    plt.legend()
    # plt.axis("equal")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()


In [2]:
import pickle
from collections import OrderedDict
from typing import Dict
from abc import abstractmethod
import numpy as np
from matplotlib import pyplot as plt
from pypcd import pypcd
import torch
import os
from pathlib import Path
from torch.utils.data import Dataset
from opencood.utils.pcd_utils import shuffle_points, mask_ego_points, downsample_lidar_minimum
from opencood.utils.transformation_utils import x1_to_x2
from opencood.data_utils.pre_processor import build_preprocessor
from opencood.data_utils.post_processor import build_postprocessor
from opencood.utils.pcd_utils import mask_points_by_range


class SingleVehicleDatasetRadar(Dataset):

    def __init__(self,params: Dict, visualize: bool = False, train: bool = True):
        self.params = params
        self.visualize = visualize
        self.train = train

        #self.save_id = 0

        # Build preprocessors
        lidar_feature_count = params['preprocess']['args']['lidar_num_point_features']
        radar_feature_count = params['preprocess']['args']['radar_num_point_features']

        self.lidar_pre_processor = build_preprocessor({**params['preprocess'], 'args': {**params['preprocess']['args'], 'num_point_features': lidar_feature_count}}, train)
        self.radar_pre_processor = build_preprocessor({**params['preprocess'], 'args': {**params['preprocess']['args'], 'num_point_features': radar_feature_count}}, train)

        self.post_processor = build_postprocessor(params["postprocess"], train)

        self.root_dir = params["root_dir"] if train else params["validate_dir"]
        print("Dataset dir:", self.root_dir)
        self.dataset_dir = os.path.dirname(self.root_dir)
        print("Dataset parent dir:", self.dataset_dir)

        if 'train_params' not in params or 'max_cav' not in params['train_params']:
            self.max_cav = 5
        else:
            self.max_cav = params['train_params']['max_cav']

        self.load_lidar_file = True if 'lidar' in params['input_source'] or self.visualize else False # TODO: also for radar but its fine for now

        self.label_type = params['label_type']
        assert self.label_type in ['lidar']

        self.generate_object_center = self.generate_object_center_lidar
        self.generate_object_center_single = self.generate_object_center

        with open(self.root_dir, 'rb') as f:
            dataset_info = pickle.load(f)
        self.dataset_info_pkl = dataset_info

        self.ego_mode = 'one'

        self.reinitialize()

        self.anchor_box = self.post_processor.generate_anchor_box()
        self.anchor_box_torch = torch.from_numpy(self.anchor_box)

        self.his_frames = params['train_params']['his_frames'] if 'his_frames' in params['train_params'] else 2
        self.fps = params['train_params']['fps'] if 'fps' in params['train_params'] else 2


    def reinitialize(self):
        self.scene_database = OrderedDict()
        if self.ego_mode == 'one':
            self.len_record = len(self.dataset_info_pkl)
            print(f'The dataset has {self.len_record} samples.')
        else:
            raise NotImplementedError(self.ego_mode)

        for i, scene_info in enumerate(self.dataset_info_pkl):
            self.scene_database.update({i: OrderedDict()})
            cav_num = scene_info['agent_num']
            assert cav_num > 0

            cav_ids = list(range(1, cav_num + 1))

            for j, cav_id in enumerate(cav_ids):
                if j > self.max_cav - 1:
                    print('too many cavs reinitialize')
                    break

                self.scene_database[i][cav_id] = OrderedDict()
                self.scene_database[i][cav_id]['ego'] = (cav_id == 1)

                sensors_dict = scene_info['agents']['1']['sensors']
                self.scene_database[i][cav_id]['sensors'] = sensors_dict
                radar_list = [k for k, v in sensors_dict.items() if v.get('sensor_type') == 'radar']
                lidar_list = [k for k, v in sensors_dict.items() if v.get('sensor_type') == 'lidar']
                self.scene_database[i][cav_id]['radar_sensors'] = radar_list
                self.scene_database[i][cav_id]['lidar_sensors'] = lidar_list


                self.scene_database[i][cav_id]['params'] = OrderedDict()
                self.scene_database[i][cav_id]['params']['vehicles'] = scene_info[f'labels']['gt_boxes_global']
                self.scene_database[i][cav_id]['params']['object_ids'] = scene_info[f'labels']['gt_object_ids'].tolist()

                self.scene_database[i][cav_id]['params']['ego_pose'] = scene_info['agents']['1']['ego_pose']['transform']
                self.scene_database[i][cav_id]['params']['ego_speed'] = scene_info['agents']['1']['ego_motion_chassis']



    def __len__(self) -> int:
        return self.len_record


    @abstractmethod
    def __getitem__(self, idx):
        base_data_dict = self.retrieve_base_data(idx)
        base_data_dict = self.retrieve_his_data(idx)
        if self.train:
            reformat_data_dict = self.get_item_train(base_data_dict)
        else:
            reformat_data_dict = self.get_item_test(base_data_dict, idx)

        return reformat_data_dict


    def get_item_train(self, base_data_dict):
        processed_data_dict = OrderedDict()

        ego_base = None
        for cav_id, cav_content in base_data_dict.items():
            if cav_content['ego']:
                ego_base = cav_content
                break
        assert ego_base is not None

        selected_cav_processed = self.get_item_single_car(ego_base)

        # No Cars
        if selected_cav_processed is None:
            return None

        processed_data_dict.update({"ego": selected_cav_processed})
        return processed_data_dict


    def get_item_test(self, base_data_dict, idx):
        processed_data_dict = OrderedDict()

        ego_id = -1
        ego_pose = []
        ego_base = None

        for cav_id, cav_content in base_data_dict.items():
            if cav_content['ego']:
                ego_id = cav_id
                ego_pose = cav_content['params']['ego_pose']
                ego_base = cav_content
                break

        assert ego_id != -1
        assert len(ego_pose) > 0

        transformation_matrix = x1_to_x2(ego_pose, ego_pose)

        ego_processed = self.get_item_single_car(ego_base)

        # No Cars
        if ego_processed is None:
            return None

        ego_processed.update({
            'transformation_matrix': transformation_matrix,
            'idx': idx,
            'cav_list': ['ego']
        })

        processed_data_dict['ego'] = ego_processed
        return processed_data_dict


    def retrieve_base_data(self, idx):
        data = OrderedDict()
        scene = self.scene_database[idx]

        ego_cav_id = None
        for cav_id, cav_content in scene.items():
            if cav_content['ego']:
                ego_cav_id = cav_id
                break
        assert ego_cav_id is not None

        cav_content = scene[ego_cav_id]
        data[f'{ego_cav_id}'] = OrderedDict()
        data[f'{ego_cav_id}']['ego'] = True
        data[f'{ego_cav_id}']['params'] = cav_content['params']

        # --------------------------------------------------------------------------------------------------------------

        # --- RADAR ----------------------------------------------------------------------------------------------------
        radar_points = []
        radar_transforms = []
        for sensor_name in cav_content['radar_sensors']:
            s = cav_content['sensors'][sensor_name]

            old_sensor_path = s['sensor_path']
            sensor_path = self.find_sensor_path(old_sensor_path)

            # --- VELOCITY PROCESSING ----------------------------------------------------------------------------------
            radar_np = self.pcd_to_npy_array(sensor_path) # local coordinate
            radar_transform = np.asarray(s['sensor_pose'])
            lidar_velocity_xyz = np.array([cav_content['params']['ego_speed']['vx'],cav_content['params']['ego_speed']['vy'],cav_content['params']['ego_speed']['vz']])
            lidar_transform = np.array(cav_content['params']['ego_pose'])

            pts = self.process_all_radar_velocity(radar_np, radar_transform, lidar_velocity_xyz, lidar_transform)

            # ----------------------------------------------------------------------------------------------------------
            # Sensor -> Ego
            T = np.asarray(s['T_sensor_to_ego'], dtype=np.float64)  # (4,4)
            xyz = pts[:, :3]
            xyz_h = np.concatenate([xyz, np.ones((xyz.shape[0], 1))], axis=1)
            xyz_ego = (T @ xyz_h.T).T[:, :3]
            pts[:, :3] = xyz_ego

            radar_points.append(pts)
            radar_transforms.append(radar_transform)
        # --------------------------------------------------------------------------------------------------------------

        # --- LIDAR ----------------------------------------------------------------------------------------------------
        lidar_points = []
        for sensor_name in cav_content['lidar_sensors']:
            s = cav_content['sensors'][sensor_name]
            old_sensor_path = s['sensor_path']
            sensor_path = self.find_sensor_path(old_sensor_path)
            pts = self.pcd_to_npy_array_lidar(sensor_path)
            T = np.asarray(s['T_sensor_to_ego'], dtype=np.float64)

            # Sensor -> Ego
            xyz = pts[:, :3]
            xyz_h = np.concatenate([xyz, np.ones((xyz.shape[0], 1))], axis=1)
            xyz_ego = (T @ xyz_h.T).T[:, :3]
            pts[:, :3] = xyz_ego

            lidar_points.append(pts)
        # --------------------------------------------------------------------------------------------------------------
        data[f'{ego_cav_id}']['lidar_np'] = np.vstack(lidar_points)
        data[f'{ego_cav_id}']['radar_np'] = np.vstack(radar_points)
        visualize_dynamic_points(data[f'{ego_cav_id}']['radar_np'], title="Radar Points with Dynamic Highlighted")
        visualize_xy_points(data[f'{ego_cav_id}']['lidar_np'], title="lidar_np XY View")
        plot_velocity_hist(data[f'{ego_cav_id}']['radar_np'], bins=50)

        return data

    def retrieve_his_data(self, idx):
        data = OrderedDict()
        scene = self.scene_database[idx]

        ego_cav_id = None
        for cav_id, cav_content in scene.items():
            if cav_content['ego']:
                ego_cav_id = cav_id
                break
        assert ego_cav_id is not None

        cav_content = scene[ego_cav_id]
        data[f'{ego_cav_id}'] = OrderedDict()
        data[f'{ego_cav_id}']['ego'] = True
        data[f'{ego_cav_id}']['params'] = cav_content['params']




        # --------------------------------------------------------------------------------------------------------------

        # --- RADAR ----------------------------------------------------------------------------------------------------
        radar_points = []
        for sensor_name in cav_content['radar_sensors']:
            s = cav_content['sensors'][sensor_name]

            old_sensor_path = s['sensor_path']
            sensor_path = self.find_sensor_path(old_sensor_path)

            # --- VELOCITY PROCESSING ----------------------------------------------------------------------------------
            radar_np = self.pcd_to_npy_array(sensor_path) # local coordinate
            radar_transform = np.asarray(s['sensor_pose'])
            lidar_velocity_xyz = np.array([cav_content['params']['ego_speed']['vx'],cav_content['params']['ego_speed']['vy'],cav_content['params']['ego_speed']['vz']])
            lidar_transform = np.array(cav_content['params']['ego_pose'])

            pts = self.process_all_radar_velocity(radar_np, radar_transform, lidar_velocity_xyz, lidar_transform)

            all_his_pts = []
            for i in range(self.his_frames):
                if idx - (i+1) < 0:
                    continue
                his_scene = self.scene_database[idx - (i+1)]
                his_cav_content = his_scene[ego_cav_id]
                his_s = his_cav_content['sensors'][sensor_name]
                his_old_sensor_path = his_s['sensor_path']
                his_sensor_path = self.find_sensor_path(his_old_sensor_path)
                his_radar_np = self.pcd_to_npy_array(his_sensor_path) # local coordinate
                his_radar_transform = np.asarray(his_s['sensor_pose'])
                his_lidar_velocity_xyz = np.array([his_cav_content['params']['ego_speed']['vx'],his_cav_content['params']['ego_speed']['vy'],his_cav_content['params']['ego_speed']['vz']])
                his_lidar_transform = np.array(his_cav_content['params']['ego_pose'])
                his_pts = self.process_his_radar_velocity(his_radar_np, his_radar_transform, his_lidar_velocity_xyz, his_lidar_transform, radar_transform, i+1)
                all_his_pts.append(his_pts)


            # ----------------------------------------------------------------------------------------------------------
            # Sensor -> Ego
            pts = np.concatenate([pts] + all_his_pts, axis=0)
            T = np.asarray(s['T_sensor_to_ego'], dtype=np.float64)  # (4,4)
            xyz = pts[:, :3]
            xyz_h = np.concatenate([xyz, np.ones((xyz.shape[0], 1))], axis=1)
            xyz_ego = (T @ xyz_h.T).T[:, :3]
            pts[:, :3] = xyz_ego

            radar_points.append(pts)
        # --------------------------------------------------------------------------------------------------------------

        # --- LIDAR ----------------------------------------------------------------------------------------------------
        lidar_points = []
        for sensor_name in cav_content['lidar_sensors']:
            s = cav_content['sensors'][sensor_name]
            old_sensor_path = s['sensor_path']
            root = Path(self.dataset_dir).name
            p = Path(old_sensor_path.strip())
            i = p.parts.index(root)
            sensor_path = Path(self.dataset_dir) / Path(*p.parts[i+1:])
            pts = self.pcd_to_npy_array_lidar(sensor_path)
            T = np.asarray(s['T_sensor_to_ego'], dtype=np.float64)

            # Sensor -> Ego
            xyz = pts[:, :3]
            xyz_h = np.concatenate([xyz, np.ones((xyz.shape[0], 1))], axis=1)
            xyz_ego = (T @ xyz_h.T).T[:, :3]
            pts[:, :3] = xyz_ego

            lidar_points.append(pts)
        # --------------------------------------------------------------------------------------------------------------
        data[f'{ego_cav_id}']['lidar_np'] = np.vstack(lidar_points)
        data[f'{ego_cav_id}']['radar_np'] = np.vstack(radar_points)
        visualize_dynamic_points(data[f'{ego_cav_id}']['radar_np'], title="Radar Points with Dynamic Highlighted")

        return data


    def generate_object_center_lidar(self, cav_contents, reference_lidar_pose):
        return self.post_processor.generate_object_center_v2x(cav_contents, reference_lidar_pose)


    def get_item_single_car(self, selected_cav_base):
        selected_cav_processed = {}

        object_bbx_center, object_bbx_mask, object_ids = self.generate_object_center_single([selected_cav_base], selected_cav_base["params"]["ego_pose"])

        # No Cars
        if len(object_ids) == 0:
            print("No Cars in the scene")
            return None

        # --- LIDAR ----------------------------------------------------------------------------------------------------
        if self.load_lidar_file or self.visualize:
            lidar_np = selected_cav_base['lidar_np']
            lidar_np = shuffle_points(lidar_np)
            lidar_np = mask_points_by_range(lidar_np, self.params['preprocess']['cav_lidar_range'])
            lidar_np = mask_ego_points(lidar_np)

            lidar_dict = self.lidar_pre_processor.preprocess(lidar_np)
            selected_cav_processed.update({'processed_lidar': lidar_dict})
        # --------------------------------------------------------------------------------------------------------------

        # --- RADAR ----------------------------------------------------------------------------------------------------
        if self.load_lidar_file or self.visualize:
            radar_np = selected_cav_base['radar_np']
            radar_np = shuffle_points(radar_np)
            radar_np = mask_points_by_range(radar_np, self.params['preprocess']['cav_lidar_range'])
            radar_np = mask_ego_points(radar_np)

            ## Speicherpfad zusammensetzen
            #save_path = f"/home/ws-ids-es3-01/PycharmProjects/hamdard_bm2cp/opencood/bilder/{self.save_id}.png"
            #self.save_radar_bev_png(radar_np, save_path)
            #self.save_id += 1  # ID hochzählen

            radar_dict = self.radar_pre_processor.preprocess(radar_np)
            selected_cav_processed.update({'processed_radar': radar_dict})



        if self.visualize:
            selected_cav_processed.update({'origin_lidar': lidar_np})

        selected_cav_processed.update(
            {
                "object_bbx_center": object_bbx_center,
                "object_bbx_mask": object_bbx_mask,
                "object_ids": object_ids,
            }
        )

        label_dict = self.post_processor.generate_label(gt_box_center=object_bbx_center, anchors=self.anchor_box, mask=object_bbx_mask)
        selected_cav_processed.update({"label_dict": label_dict})

        return selected_cav_processed


    def collate_batch_train(self, batch):
        batch = [b for b in batch if b is not None]
        if len(batch) == 0:
            return None

        output_dict = {'ego': {}}

        object_bbx_center = []
        object_bbx_mask = []
        processed_lidar_list = []
        processed_radar_list = []
        label_dict_list = []
        origin_lidar = []

        for i in range(len(batch)):
            ego_dict = batch[i]['ego']
            object_bbx_center.append(ego_dict['object_bbx_center'])
            object_bbx_mask.append(ego_dict['object_bbx_mask'])
            label_dict_list.append(ego_dict['label_dict'])

            if self.visualize:
                origin_lidar.append(ego_dict['origin_lidar'])

        object_bbx_center = torch.from_numpy(np.array(object_bbx_center))
        object_bbx_mask = torch.from_numpy(np.array(object_bbx_mask))
        label_torch_dict = self.post_processor.collate_batch(label_dict_list)

        # for centerpoint
        label_torch_dict.update({'object_bbx_center': object_bbx_center,
                                 'object_bbx_mask': object_bbx_mask})

        output_dict['ego'].update({'object_bbx_center': object_bbx_center,
                                   'object_bbx_mask': object_bbx_mask,
                                   'anchor_box': torch.from_numpy(self.anchor_box),
                                   'label_dict': label_torch_dict})
        if self.visualize:
            origin_lidar = np.array(downsample_lidar_minimum(pcd_np_list=origin_lidar))
            origin_lidar = torch.from_numpy(origin_lidar)
            output_dict['ego'].update({'origin_lidar': origin_lidar})


        if self.load_lidar_file:
            for i in range(len(batch)):
                processed_lidar_list.append(batch[i]['ego']['processed_lidar'])
            processed_lidar_torch_dict = self.lidar_pre_processor.collate_batch(processed_lidar_list)
            output_dict['ego'].update({'processed_lidar': processed_lidar_torch_dict})

        # --- RADAR ----------------------------------------------------------------------------------------------------
        if self.load_lidar_file:
            for i in range(len(batch)):
                processed_radar_list.append(batch[i]['ego']['processed_radar'])
            processed_radar_torch_dict = self.radar_pre_processor.collate_batch(processed_radar_list)
            output_dict['ego'].update({'processed_radar': processed_radar_torch_dict})

        return output_dict


    def collate_batch_test(self, batch):
        batch = [b for b in batch if b is not None]
        if len(batch) == 0:
            return None

        assert len(batch) <= 1, "Batch size 1 is required during testing!"
        batch = batch[0]

        cav_id = 'ego'
        cav_content = batch[cav_id]

        output_dict = {cav_id: {}}

        object_bbx_center = torch.from_numpy(np.array([cav_content['object_bbx_center']]))
        object_bbx_mask = torch.from_numpy(np.array([cav_content['object_bbx_mask']]))
        object_ids = cav_content['object_ids']

        output_dict[cav_id].update({"anchor_box": self.anchor_box_torch})

        if self.load_lidar_file:
            processed_lidar_torch_dict = self.lidar_pre_processor.collate_batch([cav_content['processed_lidar']])
            output_dict[cav_id].update({'processed_lidar': processed_lidar_torch_dict})

        # --- RADAR ----------------------------------------------------------------------------------------------------
        if self.load_lidar_file:
            processed_radar_torch_dict = self.radar_pre_processor.collate_batch([cav_content['processed_radar']])
            output_dict[cav_id].update({'processed_radar': processed_radar_torch_dict})


        label_torch_dict = self.post_processor.collate_batch([cav_content['label_dict']])
        label_torch_dict.update({
            'object_bbx_center': object_bbx_center,
            'object_bbx_mask': object_bbx_mask
        })

        tm = torch.from_numpy(np.array(cav_content['transformation_matrix'])).float()

        output_dict[cav_id].update({
            'object_bbx_center': object_bbx_center,
            'object_bbx_mask': object_bbx_mask,
            'label_dict': label_torch_dict,
            'object_ids': object_ids,
            'transformation_matrix': tm,
        })

        if self.visualize:
            projected_lidar_list = [cav_content['origin_lidar']]
            projected_lidar_stack = [torch.from_numpy(np.vstack(projected_lidar_list))]
            output_dict['ego'].update({'origin_lidar': projected_lidar_stack})

        return output_dict


    def post_process(self, data_dict, output_dict):
        pred_box_tensor, pred_score = self.post_processor.post_process(data_dict, output_dict)
        gt_box_tensor = self.post_processor.generate_gt_bbx(data_dict)

        return pred_box_tensor, pred_score, gt_box_tensor

    def post_process_no_fusion(self, data_dict, output_dict_ego):
        data_dict_ego = OrderedDict()
        data_dict_ego["ego"] = data_dict["ego"]
        gt_box_tensor = self.post_processor.generate_gt_bbx(data_dict)

        pred_box_tensor, pred_score = self.post_processor.post_process(data_dict_ego, output_dict_ego)
        return pred_box_tensor, pred_score, gt_box_tensor

    def post_process_no_fusion_uncertainty(self, data_dict, output_dict_ego):
        data_dict_ego = OrderedDict()
        data_dict_ego['ego'] = data_dict['ego']
        gt_box_tensor = self.post_processor.generate_gt_bbx(data_dict)

        pred_box_tensor, pred_score, uncertainty = self.post_processor.post_process(data_dict_ego, output_dict_ego, return_uncertainty=True)
        return pred_box_tensor, pred_score, gt_box_tensor, uncertainty


    # ---LIDAR ---------------------------------------------------------------------------------------------------------
    @staticmethod
    def pcd_to_npy_array_lidar(pcd_path):
        lidar = pypcd.PointCloud.from_path(pcd_path)
        pc = lidar.pc_data
        points = np.array([
            pc["x"],
            pc["y"],
            pc["z"],
            pc["intensity"]
        ], dtype=np.float64).T
        return points

    # ------------------------------------------------------------------------------------------------------------------

    # --- RADAR --------------------------------------------------------------------------------------------------------
    @staticmethod
    def pcd_to_npy_array(pcd_path):
        radar = pypcd.PointCloud.from_path(pcd_path)
        radar_data = radar.pc_data
        points = np.array([
            radar_data["x"],
            radar_data["y"],
            radar_data["z"],
            radar_data["vrel_x"],
            radar_data["vrel_y"],
            radar_data["vrel_z"],
            # radar_data["rcs"]
        ], dtype=np.float64).T
        return points


    @staticmethod
    def visualize_radar_bev(radar_np):
        x = radar_np[:, 0]
        y = radar_np[:, 1]
        color = radar_np[:, 3]  # 4th column for color

        plt.figure(figsize=(8, 8))
        sc = plt.scatter(x, y, c=color, cmap='jet', s=1)
        plt.colorbar(sc, label='Relative Velocity')
        plt.xlabel('X (meters)')
        plt.ylabel('Y (meters)')
        plt.title('Radar Point Cloud BEV (Color: Relative Velocity)')
        plt.axis('equal')
        plt.show()

    @staticmethod
    def save_radar_bev_png(radar_np, save_path):
        x = radar_np[:, 0]
        y = radar_np[:, 1]
        color = radar_np[:, 3]  # 4th column for color

        # Masken für Rot und Grau
        mask_red = (color > 0.8) | (color < -0.8)
        mask_gray = ~mask_red

        plt.figure(figsize=(8, 8))
        # Grau zeichnen
        plt.scatter(x[mask_gray], y[mask_gray], color='gray', s=1)
        # Rot zeichnen
        plt.scatter(x[mask_red], y[mask_red], color='red', s=1)

        plt.xlabel('X (meters)')
        plt.ylabel('Y (meters)')
        plt.title('Radar Point Cloud BEV (Red: |Rel. Vel| > 0.5, Gray: else)')
        plt.axis('equal')
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()

    @staticmethod
    def process_all_radar_velocity(radar_np, radar_transform, lidar_velocity_xyz, lidar_transform):
        # TODO: roll, yaw, pitch hinzufügen (kurven)

        l2r_transform = x1_to_x2(lidar_transform, radar_transform)
        l2r_rotation_matrix = l2r_transform[:3, :3]
        radar_velocity_xyz = np.dot(l2r_rotation_matrix, lidar_velocity_xyz)

        radar_np = radar_np.copy()

        r = np.sqrt(radar_np[:, 0] ** 2 + radar_np[:, 1] ** 2)
        r_safe = np.where(r == 0, 1, r)
        ux = radar_np[:, 0] / r_safe
        uy = radar_np[:, 1] / r_safe
        uz = radar_np[:, 2] / r_safe

        v_rel_vec = radar_np[:, 3:6]
        v_rel = v_rel_vec[:, 0] * ux + v_rel_vec[:, 1] * uy + v_rel_vec[:, 2] * uz

        # Compute radial speed from ego motion and sum with relative velocity
        v_ego_radial = radar_velocity_xyz[0] * ux + radar_velocity_xyz[1] * uy + radar_velocity_xyz[2] * uz
        v_r = v_rel + v_ego_radial

        # Decompose radial speed into x and y components
        beta = np.arctan2(radar_np[:, 1], radar_np[:, 0])
        v_r_x = np.cos(beta) * v_r
        v_r_y = np.sin(beta) * v_r

        # Normalize the computed velocities (clip to [-12.5, 12.5] and scale to [-1, 1])
        v_rel_norm = np.clip(v_rel, -12.5, 12.5) / 12.5
        v_r_norm = np.clip(v_r, -12.5, 12.5) / 12.5
        v_r_x_norm = np.clip(v_r_x, -12.5, 12.5) / 12.5
        v_r_y_norm = np.clip(v_r_y, -12.5, 12.5) / 12.5

        #result_np = np.column_stack(
        #    (radar_np[:, 0], radar_np[:, 1], radar_np[:, 2], v_rel_norm, v_r_norm, v_r_x_norm, v_r_y_norm))
        #result_np = np.column_stack((radar_np[:, 0], radar_np[:, 1], radar_np[:, 2], v_r_norm))
        result_np = np.column_stack((radar_np[:, 0], radar_np[:, 1], radar_np[:, 2], v_r))
        return result_np

    
    def process_his_radar_velocity(self, his_radar_np, his_radar_transform, his_lidar_velocity_xyz, his_lidar_transform, radar_transform, d_idx):
        # TODO: roll, yaw, pitch hinzufügen (kurven)

        l2r_transform = x1_to_x2(his_lidar_transform, his_radar_transform)
        l2r_rotation_matrix = l2r_transform[:3, :3]
        radar_velocity_xyz = np.dot(l2r_rotation_matrix, his_lidar_velocity_xyz)

        his_radar_np = his_radar_np.copy()

        r = np.linalg.norm(his_radar_np[:, :3], axis=1)
        r_safe = np.where(r == 0, 1e-6, r)
        ux = his_radar_np[:, 0] / r_safe
        uy = his_radar_np[:, 1] / r_safe
        uz = his_radar_np[:, 2] / r_safe
        unit_vec = his_radar_np[:, :3] / r_safe[:, None]

        v_rel_vec = his_radar_np[:, 3:6]
        v_rel = v_rel_vec[:, 0] * ux + v_rel_vec[:, 1] * uy + v_rel_vec[:, 2] * uz

        # Compute radial speed from ego motion and sum with relative velocity
        v_ego_radial = radar_velocity_xyz[0] * ux + radar_velocity_xyz[1] * uy + radar_velocity_xyz[2] * uz
        v_r = v_rel + v_ego_radial

        # doppler compensation for dynamic points
        d_t = d_idx * (1/self.fps)
        dynamic_mask = np.abs(v_r) > 0.5
        if np.any(dynamic_mask):
            delta_pos = (v_r[dynamic_mask] * d_t)[:, None] * unit_vec[dynamic_mask]
            his_radar_np[dynamic_mask, :3] += delta_pos

        # Transform historical radar points to current frame
        his2cur = x1_to_x2(his_radar_transform, radar_transform)
        his_radar_np[:, :3] = (his2cur[:3, :3] @ his_radar_np[:, :3].T).T + his2cur[:3, 3]

        # Decompose radial speed into x and y components
        beta = np.arctan2(his_radar_np[:, 1], his_radar_np[:, 0])
        v_r_x = np.cos(beta) * v_r
        v_r_y = np.sin(beta) * v_r

        # Normalize the computed velocities (clip to [-12.5, 12.5] and scale to [-1, 1])
        v_rel_norm = np.clip(v_rel, -12.5, 12.5) / 12.5
        v_r_norm = np.clip(v_r, -12.5, 12.5) / 12.5
        v_r_x_norm = np.clip(v_r_x, -12.5, 12.5) / 12.5
        v_r_y_norm = np.clip(v_r_y, -12.5, 12.5) / 12.5

        #result_np = np.column_stack(
        #    (radar_np[:, 0], radar_np[:, 1], radar_np[:, 2], v_rel_norm, v_r_norm, v_r_x_norm, v_r_y_norm))
        #result_np = np.column_stack((radar_np[:, 0], radar_np[:, 1], radar_np[:, 2], v_r_norm))
        result_np = np.column_stack((his_radar_np[:, 0], his_radar_np[:, 1], his_radar_np[:, 2], v_r))
        return result_np
    
    def find_sensor_path(self, old_sensor_path):
        root = Path(self.dataset_dir).name
        p = Path(old_sensor_path.strip())
        i = p.parts.index(root)
        sensor_path = Path(self.dataset_dir) / Path(*p.parts[i+1:])
        return sensor_path

    # ------------------------------------------------------------------------------------------------------------------

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# from opencood.data_utils.datasets.intermediate_fusion_dataset_adver_v2 import IntermediateFusionDatasetAdverV2
from opencood.hypes_yaml.yaml_utils import load_yaml
from opencood.models.point_pillar import PointPillar
import torch
def move_to_device(data, device):
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: move_to_device(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_device(x, device) for x in data]
    else:
        return data

params = load_yaml('/home/kevin/hamdard_bm2cp/opencood/hypes_yaml/v2xsim/lidar_only_with_noise/pointpillar_single.yaml')
dataset = SingleVehicleDatasetRadar(params, visualize=False)
batch_list = [dataset[90]]
data_dict = dataset.collate_batch_train(batch_list)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(params['model']['args']['where2comm_fusion'])
model = PointPillar(params['model']['args']).to(device)
data_dict = move_to_device(data_dict['ego'], device)
output = model(data_dict)

ModuleNotFoundError: No module named 'opencood.models.point_pillar'